In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit

#Files and number of header lines to skip
files = {
    "0": ("../data/data0.txt", 2),          #unstretched, no mass
    "100": ("../data/data100.txt", 3),
    "201": ("../data/data201.txt", 3),
    "301": ("../data/data301.txt", 2),
}

#Reading Equilibrium Position
t, y, err_y = {}, {}, {}
for mass, (file, skip) in files.items():
    t[mass], y[mass], err_y[mass] = np.loadtxt(file, skiprows=skip, unpack=True)

#Averaging, Uncertainties and Err Propagation
y_mean, u = {}, {}
for mass in files:
    y_mean[mass] = np.mean(y[mass])
    y_std = np.std(y[mass])
    err_y_mean = np.mean(err_y[mass])
    u[mass] = np.sqrt(y_std**2 + err_y_mean**2)
    print(f'Averaged Equilibrium of {mass} From the Sensor = {y_mean[mass]:.1f}±{u[mass]:.1f}')


In [ ]:
#Masses (kg) and gravity (m/s^2)
m = {"0": 0, "100": 0.100, "201": 0.201, "301": 0.301}
g = 9.81

#Stretched Length From the Unstretched Position (m) and mg (N)
x, err_x, F = [], [], []
for mass in files:
    x.append((y_mean["0"] - y_mean[mass]) / 100)
    err_x.append(np.sqrt(u["0"]**2 + u[mass]**2) / 100)
    F.append(m[mass] * g)
x, err_x, F = np.array(x), np.array(err_x), np.array(F)

#Linear Model
def linear(x, k, b):
    return k * x + b

#Fit (first without errors, then turn the x errors into mg errors using the slope)
popt, pcov = curve_fit(linear, x, F)
err_F = popt[0] * err_x
popt, pcov = curve_fit(linear, x, F, sigma=err_F, absolute_sigma=True)
k, b = popt
err_k, err_b = np.sqrt(np.diag(pcov))
print(f'k = {k:.1f}±{err_k:.1f} N/m')
print(f'b = {b:.2f}±{err_b:.2f} N')

#Plot
plt.errorbar(x, F, xerr=err_x, fmt='o', capsize=3, label='Data')
x_fit = np.linspace(0, max(x), 100)
plt.plot(x_fit, linear(x_fit, k, b), label=f'Fit: k = {k:.1f}±{err_k:.1f} N/m')
plt.xlabel('Stretched Length (m)')
plt.ylabel('mg (N)')
plt.legend()
plt.show()
